<!-- OPS:MAT-02:inicio -->
## Esto es “completo” hoy (55 min)

Marca con lo que realmente ocurrió. No hace falta terminar todo.

- [ ] El entorno funciona o abrí `la solución docente (no incluida)` y lo declaré.
- [ ] Producto mínimo: Una señal con umbral numérico y una acción del monitor.
- [ ] Si algo falló: una frase del error, no el traceback completo.
- [ ] Bitácora escrita, aunque diga `lab incompleto`.

Completo ≠ entendido del todo. Incompleto ≠ no puedes seguir. Si la instalación falla después de 10 minutos, usa `la solución docente (no incluida)`, conserva el artefacto y marca incompleto.
**Extensión:** Si queda tiempo: dashboard y más de dos señales; no se automatiza el reentrenamiento.
<!-- OPS:MAT-02:fin -->

# L25 · Monitoreo de supuestos en producción

**S25 · Lectura: Cap. 30 · 40–55 min**

## La pregunta

¿El servicio responde, pero el efecto sigue siendo válido?

## Qué harás

Vigilar drift, asignación, overlap y estabilidad del efecto.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Métricas por ventana y acciones ante alertas.

> **Idea clave:** Un 200 OK habla del software, no de la identificación.

La solución está en `la solución docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
from lab_assumptions import asmd, psi as psi_ref, ks_stat, auc_score
import numpy as np, pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
rng = setup()
REPS = R(10, 40)

## 1. Las cuatro métricas y qué supuesto vigila cada una
| # | Métrica | Qué supuesto protege | Cómo falla la versión ingenua |
|---|---|---|---|
| (i) | Drift de covariables | Solapamiento / validez del modelo | KS con p-valor: rechaza trivialidades con ventanas grandes y no ve nada con ventanas chicas |
| (ii) | Mecanismo de asignación | **No confusión** | Puede cambiar la regla sin que se mueva ninguna marginal |
| (iii) | Fuerza del instrumento | Relevancia | El efecto puntual parece estable aunque la inferencia ya no valga |
| (iv) | Estabilidad del efecto | El resultado que se reporta | «¿Cae dentro del IC original?» no es un test |

El orden de gravedad que da el capítulo: (i) señal temprana y a menudo benigna; (ii) ataca la identificación aunque el efecto aún no se mueva; (iii) invalida la inferencia; (iv) señal tardía.

## 2. Los datos y la ventana de referencia
Referencia = entrenamiento de IHDP-medium (1 200). El «flujo» de producción sale de validación + prueba (800 unidades), de donde **nosotros definimos la deriva** (en el repositorio la inyecta una función ya hecha). Como en L24, reconstruimos los resultados potenciales desde el efecto individual verdadero para poder cambiar la regla de asignación y seguir teniendo un `Y` coherente.

## Sección de evaluación reservada

La edición pública termina antes de las celdas que requieren `truth`, `y0`, `y1` o `tau` simulados. La evaluación docente se ejecuta en un entorno separado. Continúa la interpretación con datos observados, incertidumbre y supuestos declarados.
